# AKT1 Screening: Build/Freeze Hybrid Model + Predict Active Compounds


## Setup


In [ ]:
!pip install rdkit joblib pandas numpy scikit-learn tqdm -q

import warnings
warnings.filterwarnings("ignore")

import datetime
import numpy as np
import pandas as pd
import joblib
from pathlib import Path
from tqdm import tqdm

from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import Descriptors, rdFingerprintGenerator, MACCSkeys
from rdkit.Chem.MolStandardize import rdMolStandardize
from sklearn.base import clone, BaseEstimator, ClassifierMixin, RegressorMixin
from sklearn.model_selection import GroupKFold
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import matthews_corrcoef, balanced_accuracy_score, f1_score

RDLogger.DisableLog("rdApp.*")
_LARGEST_FRAGMENT_CHOOSER = rdMolStandardize.LargestFragmentChooser()

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False


## Redefine custom ensemble classes before loading

`classifier.joblib` / `regressor.joblib` may contain a custom `ProbaAverageEnsembleClassifier` /
`WeightedBlendRegressor` if that's what won model selection. These need to be importable in
**this** session before `joblib.load()` works. If a different model won, these are simply unused.


In [ ]:
class ProbaAverageEnsembleClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, estimator_a=None, estimator_b=None):
        self.estimator_a = estimator_a
        self.estimator_b = estimator_b

    def fit(self, X, y, sample_weight=None):
        if sample_weight is not None:
            self.estimator_a_ = clone(self.estimator_a).fit(X, y, sample_weight=sample_weight)
            self.estimator_b_ = clone(self.estimator_b).fit(X, y, sample_weight=sample_weight)
        else:
            self.estimator_a_ = clone(self.estimator_a).fit(X, y)
            self.estimator_b_ = clone(self.estimator_b).fit(X, y)
        self.classes_ = self.estimator_a_.classes_
        return self

    def predict_proba(self, X):
        return (self.estimator_a_.predict_proba(X) + self.estimator_b_.predict_proba(X)) / 2.0

    def predict(self, X):
        p = self.predict_proba(X)
        return self.classes_[np.argmax(p, axis=1)]


class WeightedBlendRegressor(BaseEstimator, RegressorMixin):
    def __init__(self, estimator_a=None, estimator_b=None, weight_a=0.5):
        self.estimator_a = estimator_a
        self.estimator_b = estimator_b
        self.weight_a = weight_a

    def fit(self, X, y):
        self.estimator_a_ = clone(self.estimator_a).fit(X, y)
        self.estimator_b_ = clone(self.estimator_b).fit(X, y)
        return self

    def predict(self, X):
        return self.weight_a * self.estimator_a_.predict(X) + (1 - self.weight_a) * self.estimator_b_.predict(X)


## 1. Load classifier + regressor, build the hybrid bundle


In [ ]:
classifier = joblib.load("classifier.joblib")
regressor = joblib.load("regressor.joblib")

# If a previously-frozen hybrid_predictor.joblib already exists on disk, prefer it (it may already
# carry blend_weight/decision_threshold from an earlier session) -- otherwise start fresh from the
# classifier/regressor bundles just loaded.
if Path("hybrid_predictor.joblib").exists():
    hybrid = joblib.load("hybrid_predictor.joblib")
    classifier = hybrid["classifier"]
    regressor = hybrid["regressor"]
    print("Loaded existing hybrid_predictor.joblib from disk.")
else:
    hybrid = {"classifier": classifier, "regressor": regressor}
    print("No existing hybrid_predictor.joblib found -- built a fresh bundle from classifier.joblib "
          "+ regressor.joblib.")

print(f"Classifier: {classifier['model_name']} | Regressor: {regressor['model_name']}")
print(f"Already frozen? blend_weight={'blend_weight' in classifier}, "
      f"decision_threshold={'decision_threshold' in classifier}")


No existing hybrid_predictor.joblib found -- built a fresh bundle from classifier.joblib + regressor.joblib.
Classifier: XGBoost Classifier | Regressor: LightGBM Regressor
Already frozen? blend_weight=False, decision_threshold=False


## 2. Freeze blend weight (w) and decision threshold -- development data only


In [ ]:
if "blend_weight" in classifier and "decision_threshold" in classifier:
    w = classifier["blend_weight"]
    threshold = classifier["decision_threshold"]
    print(f"Already frozen -- using existing values: w={w}, threshold={threshold}. Skipping OOF sweep.")
else:
    print("Not yet frozen -- computing w/threshold via development-set OOF predictions...")
    features = joblib.load("features.pkl")
    splits = joblib.load("splits.pkl")

    tune_idx = np.concatenate([splits["train_idx"], splits["val_idx"]])
    tune_groups = np.asarray(features["scaffold"])[tune_idx]

    tune_X_clf = pd.concat([splits["X_train"], splits["X_val"]], ignore_index=True)[classifier["feature_columns"]]
    tune_X_reg = pd.concat([splits["X_train"], splits["X_val"]], ignore_index=True)[regressor["feature_columns"]]
    tune_y_class = np.concatenate([splits["y_train_class"], splits["y_val_class"]])
    tune_y_pic50 = np.concatenate([splits["y_train_pic50"], splits["y_val_pic50"]])

    ACTIVE_CUTOFF = classifier["active_pic50_cutoff"]
    INACTIVE_CUTOFF = classifier["inactive_pic50_cutoff"]
    TIER_TO_LABEL = classifier["tier_to_label"]
    CLASS_LABELS_ORDERED = classifier["class_labels_ordered"]
    y_true_bin_tune = (tune_y_pic50 >= ACTIVE_CUTOFF).astype(int)

    N_FOLDS = 5
    gkf = GroupKFold(n_splits=N_FOLDS)

    oof_clf_proba = np.zeros((len(tune_y_class), len(CLASS_LABELS_ORDERED)))
    print(f"Computing {N_FOLDS}-fold scaffold-grouped OOF classifier predictions...")
    for fold_tr, fold_va in gkf.split(tune_X_clf, tune_y_class, groups=tune_groups):
        m = clone(classifier["model"])
        y_fold = tune_y_class[fold_tr]
        try:
            sw = compute_sample_weight("balanced", y_fold)
            m.fit(tune_X_clf.iloc[fold_tr], y_fold, sample_weight=sw)
        except TypeError:
            m.fit(tune_X_clf.iloc[fold_tr], y_fold)
        oof_clf_proba[fold_va] = m.predict_proba(tune_X_clf.iloc[fold_va])

    if classifier["decision_bias"] is not None:
        oof_clf_proba = oof_clf_proba * np.exp(classifier["decision_bias"])
        oof_clf_proba = oof_clf_proba / oof_clf_proba.sum(axis=1, keepdims=True)

    oof_reg_pred = np.zeros(len(tune_y_pic50))
    print(f"Computing {N_FOLDS}-fold scaffold-grouped OOF regressor predictions...")
    for fold_tr, fold_va in gkf.split(tune_X_reg, tune_y_pic50, groups=tune_groups):
        m = clone(regressor["model"])
        m.fit(tune_X_reg.iloc[fold_tr], tune_y_pic50[fold_tr])
        oof_reg_pred[fold_va] = m.predict(tune_X_reg.iloc[fold_va])

    oof_reg_proba = np.zeros_like(oof_clf_proba)
    for i, p in enumerate(oof_reg_pred):
        if p >= ACTIVE_CUTOFF:
            oof_reg_proba[i, TIER_TO_LABEL["Active"]] = 1.0
        elif p >= INACTIVE_CUTOFF:
            oof_reg_proba[i, TIER_TO_LABEL["Intermediate"]] = 1.0
        else:
            oof_reg_proba[i, TIER_TO_LABEL["Inactive"]] = 1.0

    print(f"Development set (train+val): N={len(y_true_bin_tune)}, N_active={y_true_bin_tune.sum()}")

    # PRE-REGISTRATION -- set once, do not revisit after seeing screening/external results.
    OPT_METRIC = "mcc"  # DECISION: <-- fill in your reasoning here, before running this cell
    print(f"[PRE-REGISTERED {datetime.datetime.now().isoformat(timespec='minutes')}] "
          f"Optimizing w/threshold for '{OPT_METRIC}' on development-set OOF predictions only.")

    def score_blend(w_, thr_):
        blended = w_ * oof_clf_proba + (1 - w_) * oof_reg_proba
        p_active = blended[:, TIER_TO_LABEL["Active"]]
        y_pred = (p_active >= thr_).astype(int)
        if len(np.unique(y_true_bin_tune)) < 2:
            return np.nan
        if OPT_METRIC == "mcc":
            return matthews_corrcoef(y_true_bin_tune, y_pred)
        elif OPT_METRIC == "balanced_acc":
            return balanced_accuracy_score(y_true_bin_tune, y_pred)
        elif OPT_METRIC == "f1":
            return f1_score(y_true_bin_tune, y_pred, zero_division=0)
        raise ValueError(f"Unknown OPT_METRIC: {OPT_METRIC}")

    w_grid = np.round(np.arange(0.0, 1.0001, 0.05), 3)
    thr_grid = np.round(np.arange(0.1, 0.9001, 0.025), 3)
    sweep_rows = [{"w": w_, "threshold": thr_, OPT_METRIC: score_blend(w_, thr_)}
                  for w_ in w_grid for thr_ in thr_grid]
    sweep_df = pd.DataFrame(sweep_rows).dropna()
    best_row = sweep_df.loc[sweep_df[OPT_METRIC].idxmax()]
    w = float(best_row["w"])
    threshold = float(best_row["threshold"])
    print(f"\nBest (w, threshold) on development-set OOF predictions by {OPT_METRIC}: "
          f"w={w}, threshold={threshold}, {OPT_METRIC}={best_row[OPT_METRIC]:.4f}")

    classifier["blend_weight"] = w
    classifier["decision_threshold"] = threshold
    hybrid = {"classifier": classifier, "regressor": regressor}
    joblib.dump(hybrid, "hybrid_predictor.joblib")
    print("Frozen pipeline saved to hybrid_predictor.joblib.")


## 3. Unpack the frozen bundle for prediction


In [ ]:
clf_bundle, reg_bundle = hybrid["classifier"], hybrid["regressor"]

clf_model = clf_bundle["model"]
clf_feature_columns = clf_bundle["feature_columns"]
clf_decision_bias = clf_bundle["decision_bias"]
clf_descriptor_medians = clf_bundle["descriptor_medians"]
clf_correlation_dropped_columns = clf_bundle["correlation_dropped_columns"]
TIER_TO_LABEL = clf_bundle["tier_to_label"]
CLASS_LABELS_ORDERED = clf_bundle["class_labels_ordered"]
ACTIVE_PIC50_CUTOFF = clf_bundle["active_pic50_cutoff"]
INACTIVE_PIC50_CUTOFF = clf_bundle["inactive_pic50_cutoff"]
train_ecfp4 = np.asarray(clf_bundle["train_ecfp4"])
w = clf_bundle["blend_weight"]
threshold = clf_bundle["decision_threshold"]

reg_model = reg_bundle["model"]
reg_feature_columns = reg_bundle["feature_columns"]
reg_descriptor_medians = reg_bundle["descriptor_medians"]
reg_correlation_dropped_columns = reg_bundle["correlation_dropped_columns"]

features = joblib.load("features.pkl") if "features" not in dir() else features
descriptor_names = features["descriptor_names"]
desc_cols = features["desc_cols"]
desc_func_by_name = dict(Descriptors._descList)

print(f"Using frozen pipeline: w={w}, decision_threshold={threshold}")
print(f"Applicability-domain reference set: {len(train_ecfp4)} training compounds")


## 4. Load the screening library


In [ ]:
SMILES_COL = "SMILES"   # change if your CSV uses a different header
csv_path = "AKT1_small_pool_curated_smiles_only.csv"   # replace with your file name

input_df = pd.read_csv(csv_path)
if SMILES_COL not in input_df.columns:
    raise ValueError(f"Column '{SMILES_COL}' not found. Columns present: {list(input_df.columns)}")

smiles_list = input_df[SMILES_COL].dropna().astype(str).tolist()
print(f"Screening library: {len(smiles_list)} SMILES loaded from {csv_path}")


## 5. Standardize structures and compute features (same pipeline used throughout)


In [ ]:
def smiles_to_mol(smiles):
    mol = Chem.MolFromSmiles(str(smiles))
    if mol is None:
        return None
    return _LARGEST_FRAGMENT_CHOOSER.choose(mol)

valid_input_smiles, canonical_smiles, mols = [], [], []
for smi in tqdm(smiles_list, desc="Standardizing"):
    mol = smiles_to_mol(smi)
    if mol is None:
        continue
    valid_input_smiles.append(smi)
    canonical_smiles.append(Chem.MolToSmiles(mol, canonical=True))
    mols.append(mol)

n_failed = len(smiles_list) - len(mols)
if n_failed:
    print(f"WARNING: {n_failed} SMILES failed to parse and were dropped.")
print(f"{len(mols)} compounds standardized successfully.")


In [ ]:
_ecfp4_generator = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

def morgan_fp(mol, generator=_ecfp4_generator, n_bits=2048):
    fp = generator.GetFingerprint(mol)
    arr = np.zeros((n_bits,), dtype=np.int8)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr

def rdkit_topological_fp(mol, n_bits=2048):
    fp = Chem.RDKFingerprint(mol, fpSize=n_bits)
    arr = np.zeros((n_bits,), dtype=np.int8)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr

def maccs_fp(mol):
    fp = MACCSkeys.GenMACCSKeys(mol)
    arr = np.zeros((167,), dtype=np.int8)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr

def calc_2d_descriptors(mol):
    values = []
    for name in descriptor_names:
        func = desc_func_by_name.get(name)
        if func is None:
            values.append(np.nan)
            continue
        try:
            values.append(func(mol))
        except Exception:
            values.append(np.nan)
    return values

scr_ecfp4 = np.array([morgan_fp(m) for m in tqdm(mols, desc="ECFP4")])
scr_rdkit_topo = np.array([rdkit_topological_fp(m) for m in tqdm(mols, desc="RDKit topological")])
scr_maccs = np.array([maccs_fp(m) for m in tqdm(mols, desc="MACCS")])
scr_hybrid_fp = np.concatenate([scr_ecfp4, scr_rdkit_topo, scr_maccs], axis=1)
scr_desc_all = np.array([calc_2d_descriptors(m) for m in tqdm(mols, desc="2D descriptors")])
scr_desc_df = pd.DataFrame(scr_desc_all, columns=desc_cols).replace([np.inf, -np.inf], np.nan)
fp_cols = [f"FP_{i}" for i in range(scr_hybrid_fp.shape[1])]
X_scr_all = pd.concat([pd.DataFrame(scr_hybrid_fp, columns=fp_cols),
                       scr_desc_df.reset_index(drop=True)], axis=1)
X_scr_all_raw = X_scr_all.copy()  # pristine snapshot for the regressor pipeline below
print(f"Raw screening feature matrix: {X_scr_all.shape}")


## 6. Build classifier and regressor feature matrices


In [ ]:
X_scr_clf = X_scr_all.copy()
X_scr_clf[desc_cols] = X_scr_clf[desc_cols].fillna(clf_descriptor_medians)
X_scr_clf = X_scr_clf.drop(columns=[c for c in clf_correlation_dropped_columns if c in X_scr_clf.columns])
missing_clf_cols = [c for c in clf_feature_columns if c not in X_scr_clf.columns]
if missing_clf_cols:
    raise ValueError(f"Missing classifier feature columns: {missing_clf_cols[:10]}")
X_scr_clf_final = X_scr_clf[clf_feature_columns]
print(f"Final classifier feature matrix: {X_scr_clf_final.shape}")

X_scr_reg = X_scr_all_raw.copy()
X_scr_reg[desc_cols] = X_scr_reg[desc_cols].fillna(reg_descriptor_medians)
for col in reg_correlation_dropped_columns:
    if col in X_scr_reg.columns:
        X_scr_reg.drop(columns=[col], inplace=True)
missing_reg_cols = [c for c in reg_feature_columns if c not in X_scr_reg.columns]
if missing_reg_cols:
    raise ValueError(f"Missing regressor feature columns: {missing_reg_cols[:10]}")
X_scr_reg_final = X_scr_reg[reg_feature_columns]
print(f"Final regressor feature matrix: {X_scr_reg_final.shape}")


## 7. Predict with the hybrid model (classifier + regressor, frozen w/threshold)


In [ ]:
clf_proba = clf_model.predict_proba(X_scr_clf_final)
if clf_decision_bias is not None:
    clf_proba = clf_proba * np.exp(clf_decision_bias)
    clf_proba = clf_proba / clf_proba.sum(axis=1, keepdims=True)

predicted_pic50 = reg_model.predict(X_scr_reg_final)
reg_proba = np.zeros((len(predicted_pic50), len(CLASS_LABELS_ORDERED)))
for i, p in enumerate(predicted_pic50):
    if p >= ACTIVE_PIC50_CUTOFF:
        reg_proba[i, TIER_TO_LABEL["Active"]] = 1.0
    elif p >= INACTIVE_PIC50_CUTOFF:
        reg_proba[i, TIER_TO_LABEL["Intermediate"]] = 1.0
    else:
        reg_proba[i, TIER_TO_LABEL["Inactive"]] = 1.0

blended_proba = w * clf_proba + (1 - w) * reg_proba
active_col = TIER_TO_LABEL["Active"]
p_active = blended_proba[:, active_col]
hybrid_active_call = np.where(p_active >= threshold, "Active", "Not Active")

print(f"Predicted {len(predicted_pic50)} compounds. "
      f"Hybrid model calls {int((p_active >= threshold).sum())} Active at threshold={threshold}.")


## 8. Nearest-training-set Tanimoto similarity (applicability domain)


In [ ]:
def nearest_training_tanimoto(query_fps, train_fps):
    query_fps = np.asarray(query_fps, dtype=np.int32)
    train_fps = np.asarray(train_fps, dtype=np.int32)
    inter = query_fps @ train_fps.T
    pa = query_fps.sum(axis=1, keepdims=True)
    pb = train_fps.sum(axis=1, keepdims=True).T
    union = pa + pb - inter
    sims = np.divide(inter, union, out=np.zeros_like(inter, dtype=float), where=union > 0)
    return sims.max(axis=1)

nearest_training_sim = nearest_training_tanimoto(scr_ecfp4, train_ecfp4)
print(f"Median nearest-training Tanimoto: {np.median(nearest_training_sim):.3f}")


## 9. Assemble results and select high-confidence active compounds

High confidence = `predicted_pIC50 > 7.0` **and** `nearest_training_tanimoto > 0.6`. The hybrid
model's own Active/Not-Active call (at the frozen threshold) is included for reference but does
not itself gate this tier.


In [ ]:
results = pd.DataFrame({
    "input_smiles": valid_input_smiles,
    "canonical_smiles": canonical_smiles,
    "predicted_pIC50": predicted_pic50,
    "hybrid_active_probability": p_active,
    "hybrid_active_call": hybrid_active_call,
    "nearest_training_tanimoto": nearest_training_sim,
})

HIGH_CONF_PIC50_CUTOFF = 7.0
HIGH_CONF_TANIMOTO_CUTOFF = 0.6

results["high_confidence"] = (
    (results["predicted_pIC50"] > HIGH_CONF_PIC50_CUTOFF)
    & (results["nearest_training_tanimoto"] > HIGH_CONF_TANIMOTO_CUTOFF)
)

high_confidence = results[results["high_confidence"]].copy()
high_confidence = high_confidence.sort_values(
    ["predicted_pIC50", "nearest_training_tanimoto"], ascending=[False, False]
)

print(f"Total scored: {len(results)}")
print(f"High-confidence (pIC50 > {HIGH_CONF_PIC50_CUTOFF}, Tanimoto > {HIGH_CONF_TANIMOTO_CUTOFF}): "
      f"{len(high_confidence)}")
print(f"  ... of which the hybrid model also calls Active at threshold={threshold}: "
      f"{(high_confidence['hybrid_active_call'] == 'Active').sum()}")

high_confidence.head(20)


## 10. Save results


In [ ]:
results.to_csv("akt1_hybrid_all_predictions.csv", index=False)
high_confidence.to_csv("akt1_hybrid_high_confidence.csv", index=False)
print(f"Saved {len(results)} scored compounds to akt1_hybrid_all_predictions.csv")
print(f"Saved {len(high_confidence)} high-confidence compounds to akt1_hybrid_high_confidence.csv")

if IN_COLAB:
    files.download("akt1_hybrid_all_predictions.csv")
    files.download("akt1_hybrid_high_confidence.csv")
